<a href="https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1. My rule and its reason codes

Signal check 1 — staleness (behind the refresh flags):
Bucketing days_since_last_update against trend_pct and pct_declining:
  0-90d    (n=20,655): avg_trend_pct = +0.71%, 51.2% declining
  91-180d  (n=9,171):  avg_trend_pct = -15.68%, 61.1% declining
  181-365d (n=169):    avg_trend_pct = -4.72%, 46.7% declining
  365d+    (n=5):      avg_trend_pct = -96.17%, 60.0% declining (n too small to trust)
VERDICT: MIXED. The 91-180d bucket is clearly the worst, but decline doesn't get monotonically
worse with more staleness -- the 181-365d bucket partially recovers, and the 365d+ "worst"
bucket has only 5 rows, too few to trust. Staleness alone is not a clean, reliable signal here
-- it's a real, honest negative, not something to lean the rule's score on.

Signal check 2 — CTR vs. position (behind the CTR-fix logic):
Bucketing avg_position against mean ctr:
  position 1-3  (n=1,141):  avg_ctr = 2.714
  position 4-10 (n=11,842): avg_ctr = 0.651
  position 11-20 (n=7,273): avg_ctr = 0.323
  position 21+  (n=8,524):  avg_ctr = 0.212
VERDICT: CONFIRMED. CTR drops sharply and monotonically as position worsens -- exactly what the
CTR-fix logic assumes. This is the reliable signal, so the rule below is built on it, not on
staleness.

My rule: for pages with real traffic (impressions_90d >= 100), compute each page's expected CTR
for its position bucket (from the table above), subtract its actual CTR to get a gap, and
multiply by impressions_90d -- so a page underperforming by the same CTR gap gets a higher score
if more people are actually seeing it. Reason code: CTR_BELOW_EXPECTED_FOR_POSITION (one code,
since this is a single rule, not a multi-branch system). Action: review_snippet_meta.

I'm deliberately not using staleness as the primary driver, since signal check 1 showed it's
mixed, not confirmed -- building the score around a signal I just showed isn't reliable would be
dishonest.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/umairhussainn/ml-internship"
REPO_DIR = "ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd
pd.set_option('display.width', 120)
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# --- Signal 1: staleness vs decline ---
bins = [-1, 90, 180, 365, 100000]
labels = ['0-90d', '91-180d', '181-365d', '365d+']
df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=bins, labels=labels)
staleness_table = df.groupby('staleness_bucket', observed=True).agg(
    n=('content_id', 'count'),
    avg_trend_pct=('trend_pct', 'mean'),
    pct_declining=('trend_direction', lambda s: (s.str.lower() == 'down').mean())
).round(3)
print("SIGNAL 1 -- staleness vs decline:")
print(staleness_table)
print("VERDICT: MIXED\n")

# --- Signal 2: CTR vs position ---
pos_bins = [0, 3, 10, 20, 100]
pos_labels = ['1-3', '4-10', '11-20', '21+']
df['position_bucket'] = pd.cut(df['avg_position'], bins=pos_bins, labels=pos_labels)
ctr_table = df.groupby('position_bucket', observed=True).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean')
).round(4)
print("SIGNAL 2 -- CTR vs position:")
print(ctr_table)
print("VERDICT: CONFIRMED")

SIGNAL 1 -- staleness vs decline:
                      n  avg_trend_pct  pct_declining
staleness_bucket                                     
0-90d             20655          0.707          0.512
91-180d            9171        -15.683          0.611
181-365d            169         -4.718          0.467
365d+                 5        -96.167          0.600
VERDICT: MIXED

SIGNAL 2 -- CTR vs position:
                     n  avg_ctr
position_bucket                
1-3               1141   2.7143
4-10             11842   0.6510
11-20             7273   0.3234
21+               8524   0.2117
VERDICT: CONFIRMED


In [3]:
expected_ctr = df.groupby('position_bucket', observed=True)['ctr'].mean().astype(float).to_dict()
df['expected_ctr'] = df['position_bucket'].astype(str).map(expected_ctr).astype(float)
df['ctr_gap'] = df['expected_ctr'] - df['ctr']

MIN_IMPRESSIONS = 100
eligible = df[df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
eligible['action_score'] = (eligible['ctr_gap'] * eligible['impressions_90d']).clip(lower=0)
eligible['reason_code'] = 'CTR_BELOW_EXPECTED_FOR_POSITION'
eligible['action'] = 'review_snippet_meta'

queue = eligible.sort_values('action_score', ascending=False)[
    ['content_id', 'client_id', 'content_type', 'avg_position', 'position_bucket',
     'ctr', 'expected_ctr', 'ctr_gap', 'impressions_90d', 'action_score',
     'reason_code', 'action']
].reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
print(f"({len(eligible):,} of {len(df):,} total passed the impressions_90d >= {MIN_IMPRESSIONS} gate)")
queue.head(10)

Wrote 22,006 rows to work/outputs/baseline_action_score.csv
(22,006 of 30,000 total passed the impressions_90d >= 100 gate)


,content_id,client_id,content_type,avg_position,position_bucket,ctr,expected_ctr,ctr_gap,impressions_90d,action_score,reason_code,action
0,content_8c19996aa890,client_4e07408562,keyword article,2.5,1-3,0.15,2.714303,2.564303,509252,1.305877e+06,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
1,content_4c36c775b818,client_4e07408562,keyword article,2.3,1-3,0.41,2.714303,2.304303,463103,1.067130e+06,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
2,content_8451fc6f034d,client_d029fa3a95,keyword article,2.3,1-3,0.03,2.714303,2.684303,272144,7.305170e+05,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
3,content_44e481c8f55b,client_19581e27de,keyword article,1.4,1-3,0.65,2.714303,2.064303,312694,6.454952e+05,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
4,content_9532f197bbc8,client_4e07408562,keyword article,2.0,1-3,0.87,2.714303,1.844303,309192,5.702438e+05,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
5,content_e12868d1f396,client_4e07408562,keyword article,2.9,1-3,0.07,2.714303,2.644303,149712,3.958839e+05,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
6,content_4a6607efcb46,client_6208ef0f77,keyword article,2.2,1-3,0.01,2.714303,2.704303,128068,3.463347e+05,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
7,content_4fc39a2b8cf0,client_19581e27de,keyword article,2.6,1-3,0.69,2.714303,2.024303,160959,3.258298e+05,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
8,content_11900bd7941a,client_4e07408562,keyword article,2.8,1-3,0.41,2.714303,2.304303,123561,2.847220e+05,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta
9,content_03d2673b2553,client_19581e27de,keyword article,1.9,1-3,0.83,2.714303,1.884303,143314,2.700470e+05,CTR_BELOW_EXPECTED_FOR_POSITION,review_snippet_meta


In [4]:
top10 = queue.head(10)
for i, row in top10.iterrows():
    print(f"#{i+1} {row['content_id']} (pos {row['avg_position']}, ctr {row['ctr']:.2f} vs "
          f"expected {row['expected_ctr']:.2f}, {row['impressions_90d']:,} impressions)")

#1 content_8c19996aa890 (pos 2.5, ctr 0.15 vs expected 2.71, 509,252 impressions)
#2 content_4c36c775b818 (pos 2.3, ctr 0.41 vs expected 2.71, 463,103 impressions)
#3 content_8451fc6f034d (pos 2.3, ctr 0.03 vs expected 2.71, 272,144 impressions)
#4 content_44e481c8f55b (pos 1.4, ctr 0.65 vs expected 2.71, 312,694 impressions)
#5 content_9532f197bbc8 (pos 2.0, ctr 0.87 vs expected 2.71, 309,192 impressions)
#6 content_e12868d1f396 (pos 2.9, ctr 0.07 vs expected 2.71, 149,712 impressions)
#7 content_4a6607efcb46 (pos 2.2, ctr 0.01 vs expected 2.71, 128,068 impressions)
#8 content_4fc39a2b8cf0 (pos 2.6, ctr 0.69 vs expected 2.71, 160,959 impressions)
#9 content_11900bd7941a (pos 2.8, ctr 0.41 vs expected 2.71, 123,561 impressions)
#10 content_03d2673b2553 (pos 1.9, ctr 0.83 vs expected 2.71, 143,314 impressions)


1. review_snippet_meta -- position 2.5 but CTR 0.15 vs expected 2.71, 509K impressions.
   Wrong if: the low CTR is because the snippet already got manually rewritten last week and
   this data hasn't refreshed yet.
2. review_snippet_meta -- position 2.3, CTR 0.41 vs 2.71, 463K impressions.
   Wrong if: this page already ranks for a different, more specific query than the one driving
   the position-bucket average, making the "expected CTR" comparison apples-to-oranges.
3. review_snippet_meta -- position 2.3, CTR 0.03 vs 2.71, 272K impressions.
   Wrong if: this is a rich-result/featured-snippet page where users read the answer directly in
   search results without clicking -- low CTR there is expected behavior, not a broken snippet.
[... continue the same pattern for rows 4-10, one line each, using the printed numbers]

In [5]:
# Weak pick check: is this rule accidentally using future or label-derived info?
print("Columns used in the rule:", ['avg_position', 'ctr', 'impressions_90d'])
print("None of these are trend_direction, trend_pct, or any *_last_30d/*_prev_30d column --")
print("all are current-state signals, not future-window or label-derived. Clean.")
print()
# Weak pick: rows where the gap looks big but n behind the expected_ctr bucket is thin,
# or where competition/cpc suggests this traffic isn't even commercially relevant
weak = queue[queue['position_bucket'] == '1-3'].tail(3)
print("Weakest-looking picks in the top set (smallest gap among position 1-3 rows):")
print(weak[['content_id', 'ctr', 'expected_ctr', 'impressions_90d']])

Columns used in the rule: ['avg_position', 'ctr', 'impressions_90d']
None of these are trend_direction, trend_pct, or any *_last_30d/*_prev_30d column --
all are current-state signals, not future-window or label-derived. Clean.

Weakest-looking picks in the top set (smallest gap among position 1-3 rows):
                 content_id   ctr  expected_ctr  impressions_90d
12517  content_19cdc614df78  1.56      2.714303              128
18478  content_e0cf84281b49  5.19      2.714303             3023
21512  content_10d944e9e198  6.19      2.714303              113
